In [ ]:
!pip install s3fs
!pip install zarr
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.8/201.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 23.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.1.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.1.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 113.0 MB/s eta 0:00:00


In [ ]:
import os
from typing import Union
import numpy as np
import pandas as pd
import s3fs
import zarr
from tqdm.auto import tqdm

AWS_ZARR_ROOT = (
    "s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/2018"
)

# AWS_ZARR_ROOT = (
#     "s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2_small.zarr/"
# )

# Create S3 file system
s3 = s3fs. S3FileSystem(anon=True)  # anon=True for anonymous access (public bucket)

# Open the zarr dataset from S3
zarr_store = s3fs.S3Map(root=AWS_ZARR_ROOT, s3=s3)
dataset = zarr. open(zarr_store, mode='r')

# Access the dataset
print(f"Dataset info:\n{dataset.info}")
print(f"\nAvailable arrays: {list(dataset.arrays())}")


Dataset info:
Name        : 
Type        : Group
Zarr format : 2
Read-only   : True
Store type  : FsspecStore

Available arrays: [('1700A', <Array <FsspecStore(S3FileSystem, gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/2018)>/1700A shape=(76723, 512, 512) dtype=float32>), ('193A', <Array <FsspecStore(S3FileSystem, gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/2018)>/193A shape=(84881, 512, 512) dtype=float32>), ('1600A', <Array <FsspecStore(S3FileSystem, gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/2018)>/1600A shape=(84897, 512, 512) dtype=float32>), ('211A', <Array <FsspecStore(S3FileSystem, gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/2018)>/211A shape=(84896, 512, 512) dtype=float32>), ('131A', <Array <FsspecStore(S3FileSystem, gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/2018)>/131A shape=(84783, 512, 512) dtype=float32>), ('171A', <Array <FsspecStore(S3FileSystem, gov-nasa-hdrl-data1/con

In [ ]:
def s3_connection(path_to_zarr: str) -> s3fs.S3Map:
    """
    Create an S3 connection for zarr access
    """
    s3 = s3fs.S3FileSystem(anon=True)  # anon=True for anonymous access (public bucket)
    return s3fs.S3Map(root=path_to_zarr, s3=s3)


def load_single_aws_zarr(
    path_to_zarr: str,
    cache_max_single_size: int | None = None,
) -> zarr.Array | zarr.Group:
    """
    Load zarr from S3 using the new zarr v3 API.
    Compatible with Python 3. 12+
    """
    # In zarr v3, use the S3Map directly without LRUStoreCache
    # Caching is handled internally by zarr
    store = s3_connection(path_to_zarr)

    return zarr. open(
        store=store,
        mode="r",
    )


# Load the dataset
root = load_single_aws_zarr(
    path_to_zarr=AWS_ZARR_ROOT,
)

# Display the zarr structure
print(root.tree())

/
├── 131A (84783, 512, 512) float32
├── 1600A (84897, 512, 512) float32
├── 1700A (76723, 512, 512) float32
├── 171A (84885, 512, 512) float32
├── 193A (84881, 512, 512) float32
├── 211A (84896, 512, 512) float32
├── 304A (84883, 512, 512) float32
├── 335A (84784, 512, 512) float32
└── 94A (84899, 512, 512) float32

In [ ]:
data = root["193A"]

import dask.array as da

all_image = da.from_array(data)
all_image

#try log scale the pixel

dask.array<array, shape=(84881, 512, 512), dtype=float32, chunksize=(120, 512, 512), chunktype=numpy.ndarray>

In [ ]:
df_plasma = pd.read_csv('solarwind2018.lst.txt',
                         sep=r'\s+',
                         header=None,
                         names=['year', 'day', 'hour', 'plasma_speed'])

# 2. Convert Year + Day + Hour into a single DatetimeIndex
# %j is the format code for Day of the Year (1-366)
df_plasma['time'] = pd.to_datetime(
    df_plasma['year'].astype(str) + '-' + df_plasma['day'].astype(str) + ' ' + df_plasma['hour'].astype(str) + ':00:00',
    format='%Y-%j %H:%M:%S'
)

# 3. Set the time as the index
df_plasma.set_index('time', inplace=True)

# 4. Clean the data
# OMNI files use 999.9 or 9999.0 for missing data.
# We replace it with NaN so it doesn't ruin the CNN training.
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].replace(9999.0, np.nan)

# 5. Interpolate missing values
# This fills small gaps in the 1-hour cadence so you have continuous targets for ML
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].interpolate(method='linear')
# very few, so interpolation doesn't really matter here.

# 6. Keep only the necessary column
df_plasma = df_plasma[['plasma_speed']]

print("df_plasma created successfully!")
print(df_plasma.head())

df_plasma created successfully!
                     plasma_speed
time                             
2018-01-01 00:00:00         381.0
2018-01-01 01:00:00         406.0
2018-01-01 02:00:00         392.0
2018-01-01 03:00:00         396.0
2018-01-01 04:00:00         416.0


In [ ]:
import os
from typing import Union
import numpy as np
import pandas as pd
import s3fs
import zarr
from tqdm.auto import tqdm

df_plasma1 = pd.read_csv('omni2_sw131415.lst.txt',
                         sep=r'\s+',
                         header=None,
                         names=['year', 'day', 'hour', 'plasma_speed'])

df_plasma2 = pd.read_csv('omni2_2017.lst.txt',
                         sep=r'\s+',
                         header=None,
                         names=['year', 'day', 'hour', 'plasma_speed'])
df_plasma = pd.concat([df_plasma1, df_plasma2], ignore_index=True)

# 2. Convert Year + Day + Hour into a single DatetimeIndex
# %j is the format code for Day of the Year (1-366)
df_plasma['time'] = pd.to_datetime(
    df_plasma['year'].astype(str) + '-' + df_plasma['day'].astype(str) + ' ' + df_plasma['hour'].astype(str) + ':00:00',
    format='%Y-%j %H:%M:%S'
)

# 3. Set the time as the index
df_plasma.set_index('time', inplace=True)

# 4. Clean the data
# OMNI files use 999.9 or 9999.0 for missing data.
# We replace it with NaN so it doesn't ruin the CNN training.
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].replace(9999.0, np.nan)

# 5. Interpolate missing values
# This fills small gaps in the 1-hour cadence so you have continuous targets for ML
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].interpolate(method='linear')
# very few, so interpolation doesn't really matter here.

# 6. Keep only the necessary column
df_plasma = df_plasma[['plasma_speed']]

print("df_plasma created successfully!")
print(df_plasma.head())

df_plasma created successfully!
                     plasma_speed
time                             
2013-01-01 00:00:00         358.0
2013-01-01 01:00:00         355.0
2013-01-01 02:00:00         359.0
2013-01-01 03:00:00         352.0
2013-01-01 04:00:00         351.0


In [ ]:
mse_mean = ((df_plasma - df_plasma.mean()) ** 2).mean()
rmse = np.sqrt(mse_mean)
print("Mean baseline RMSE:", rmse)

Mean baseline RMSE: plasma_speed    96.349664
dtype: float64


In [ ]:
mse_mean = ((df_plasma - df_plasma.mean()) ** 2).mean()
print("Mean baseline MSE:", mse_mean)

Mean baseline MSE: plasma_speed    9283.257824
dtype: float64


In [ ]:
zarr_times_raw = data.attrs['DATE-OBS']
zarr_times_unsorted = pd.to_datetime(list(zarr_times_raw))

# Create a sorting index to keep track of where the data actually lives
sort_order = np.argsort(zarr_times_unsorted)
zarr_times_sorted = zarr_times_unsorted[sort_order]

# 2. Match Plasma Data to the Sorted Timestamps
# (Assuming df_plasma is already loaded and indexed by time)
def get_nearest_sorted_indices(target_times, source_times_sorted):
    indices = np.searchsorted(source_times_sorted, target_times)
    indices = np.clip(indices, 1, len(source_times_sorted) - 1)

    left = source_times_sorted[indices - 1]
    right = source_times_sorted[indices]
    use_left = (target_times - left) < (right - target_times)

    return np.where(use_left, indices - 1, indices)

AU_KM = 149_597_870.7
tof_seconds = AU_KM / df_plasma['plasma_speed']
df_plasma['tof'] = pd.to_timedelta(tof_seconds, unit='s')
df_plasma['lookup_time'] = df_plasma.index - df_plasma['tof']

sorted_match_indices = get_nearest_sorted_indices(
    df_plasma['lookup_time'].values,
    zarr_times_sorted
)

df_plasma['zarr_idx'] = sort_order[sorted_match_indices]


# 4. Filter with the same 15-minute tolerance
# We compare the target time to the actual timestamp at the matched index
matched_timestamps = zarr_times_unsorted[df_plasma['zarr_idx'].values]

time_diffs = np.abs(matched_timestamps - df_plasma['lookup_time'])

df_aligned = df_plasma[time_diffs < pd.Timedelta(minutes=15)].copy()

print(f"Dataset Ready: {len(df_aligned)} image-speed pairs.")
(df_plasma['tof'].dt.total_seconds() / 86400).describe()

NameError: name 'data' is not defined

In [ ]:
LAG_TIME = pd.Timedelta(days=4)

# 2. Extract and Sort Zarr Timestamps (as done before)
zarr_times_raw = data.attrs['DATE-OBS']
zarr_times_unsorted = pd.to_datetime(list(zarr_times_raw))
sort_order = np.argsort(zarr_times_unsorted)
zarr_times_sorted = zarr_times_unsorted[sort_order]

# 3. Apply the lag to the Plasma Data
# We create a new column 'lookup_time' which is the time the wind LEFT the sun
df_plasma['lookup_time'] = df_plasma.index - LAG_TIME

# 4. Match the LAGGED time to the Zarr images
def get_nearest_sorted_indices(target_times, source_times_sorted):
    indices = np.searchsorted(source_times_sorted, target_times)
    indices = np.clip(indices, 1, len(source_times_sorted) - 1)
    left = source_times_sorted[indices - 1]
    right = source_times_sorted[indices]
    use_left = (target_times - left) < (right - target_times)
    return np.where(use_left, indices - 1, indices)

# Match using the shifted 'lookup_time'
sorted_match_indices = get_nearest_sorted_indices(df_plasma['lookup_time'], zarr_times_sorted)
df_plasma['zarr_idx'] = sort_order[sorted_match_indices]

# 5. Filter by 15-minute tolerance
matched_timestamps = zarr_times_unsorted[df_plasma['zarr_idx'].values]
time_diffs = np.abs(matched_timestamps - df_plasma['lookup_time'])

df_aligned = df_plasma[time_diffs < pd.Timedelta(minutes=15)].copy()

print(f"Dataset Ready with 4-day lag: {len(df_aligned)} pairs.")

Dataset Ready with 4-day lag: 8492 pairs.


In [ ]:
class SolarWindDataset:
    def __init__(self, aligned_df, zarr_array):
        self.df = aligned_df
        self.images = zarr_array

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Pull image from Zarr using our saved index
        x = self.images[int(row['zarr_idx']), :, :]
        y = row['plasma_speed']

        # ML Preprocessing: Normalize and add channel dimension (1, 512, 512)
        x = np.log1p(np.maximum(x, 0))
        return x[None, :, :], y

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# 1. Corrected CNN Architecture
class SimpleSolarCNN(nn.Module):
    def __init__(self):
        super(SimpleSolarCNN, self).__init__() # Fixed: added double underscores
        self.conv1 = nn.Conv2d(1, 32, kernel_size=9, stride=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.pool3 = nn.MaxPool2d(3, 1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=2, stride=2)

        self.fc1 = nn.Linear(8 * 8 * 128, 4096)
        self.fc2 = nn.Linear(4096, 1)

        self.dropout_fc = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool3((F.relu(self.conv3(x))))
        x = x.view(x.size(0), -1) # Flatten
        x = self.dropout_fc(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 2. Split Data
train_df, val_df = train_test_split(
    df_aligned,
    test_size=0.2,      # 80% train / 20% val
    random_state=322,
    shuffle=True
)

# 3. Scale Plasma Speed
scaler = MinMaxScaler()
train_df['plasma_speed'] = scaler.fit_transform(train_df[['plasma_speed']])
val_df['plasma_speed'] = scaler.transform(val_df[['plasma_speed']])

# 4. Setup DataLoaders
train_dataset = SolarWindDataset(train_df, data)
val_dataset   = SolarWindDataset(val_df, data)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,      # shuffle ONLY for training
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,     # NEVER shuffle validation
    num_workers=2,
    pin_memory=True,
)
# 5. Initialize Model & Training Tools
model = SimpleSolarCNN()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Starting trial with {len(train_df)} training samples and {len(val_df)} validation samples using {device}...")

Starting trial with 6733 training samples and 1684 validation samples using cuda...


In [ ]:
num_epochs = 5
best_val_loss = float('inf')

for epoch in range(num_epochs):

    # ---------- TRAIN ----------
    model.train()
    train_loss = 0.0

    train_loop = tqdm(train_loader, leave=True)
    train_loop.set_description(f"Epoch [{epoch+1}/{num_epochs}] TRAIN")

    for images, speeds in train_loop:
        images = images.float().to(device)
        speeds = speeds.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, speeds)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    train_loss /= len(train_loader)

    # ---------- VALIDATION ----------
    model.eval()
    val_loss = 0.0

    val_loop = tqdm(val_loader, leave=False)
    val_loop.set_description(f"Epoch [{epoch+1}/{num_epochs}] VAL")

    with torch.no_grad():
        for images, speeds in val_loop:
            images = images.float().to(device)
            speeds = speeds.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, speeds)
            val_loss += loss.item()
            val_loop.set_postfix(loss=loss.item())

    val_loss /= len(val_loader)

    # ---------- LOG ----------
    print(f"Epoch {epoch+1}: Train {train_loss:.2f} | Val {val_loss:.2f}")

    # ---------- CHECKPOINT ----------
    ckpt = {
        'epoch': epoch,
        'model': model.state_dict(),
        'optim': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
    }

    torch.save(ckpt, "last_ckpt.pt")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        print("✅ Best model updated")

  0%|          | 0/106 [00:00<?, ?it/s]

RuntimeError: Caught HTTPClientError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/aiobotocore/httpsession.py", line 230, in send
    response = await session.request(
               ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/aiohttp/client.py", line 779, in _request
    resp = await handler(req)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/aiohttp/client.py", line 734, in _connect_and_send_request
    conn = await self._connector.connect(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/aiohttp/connector.py", line 672, in connect
    proto = await self._create_connection(req, traces, timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/aiohttp/connector.py", line 1239, in _create_connection
    _, proto = await self._create_direct_connection

## Profiling the Training Loop with `torch.profiler`

To identify performance bottlenecks, we can use `torch.profiler`. This will record various metrics for CPU and GPU operations during a few training iterations. This can help pinpoint if the bottleneck is in data loading, model computations, or I/O.

In [ ]:
import torch.profiler

# It's usually best to profile only a few iterations to avoid excessive overhead.
# Let's run a small number of training steps with the profiler.

# Re-initialize the model to ensure a fresh start for profiling if needed
# model = SimpleSolarCNN().to(device)
# criterion = nn.MSELoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


print("Starting profiling for a few training iterations...")

with torch.profiler.profile(
    schedule=torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1),
    on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/simple_solar_cnn'),
    with_stack=True
) as profiler:
    for step, (images, speeds) in enumerate(train_loader):
        if step >= (1 + 1 + 3) * 1: # Adjust based on schedule parameters
            break

        images = images.float().to(device)
        speeds = speeds.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, speeds)
        loss.backward()
        optimizer.step()

        profiler.step()

print("Profiling complete. Results saved to ./log/simple_solar_cnn for TensorBoard visualization.")

# You can also print a summary directly if you don't want to use TensorBoard
# print(profiler.key_averages().table(sort_by="cuda_time_total", row_limit=10))


Starting profiling for a few training iterations...
Profiling complete. Results saved to ./log/simple_solar_cnn for TensorBoard visualization.


In [ ]:
print(profiler.key_averages().table(sort_by="cuda_time_total", row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.01%      21.566ms        99.98%      201.438s       67.146s       0.000us         0.00%     100.707ms      33.569ms             3  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      77.536ms        45.36%      77.536ms      25.845ms             3  
         

In [ ]:
profiler.key_averages().table()

'-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  \n                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  \n-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  \nvoid at::native::elementwise_kernel<128, 2, at::nati...         0.00%       0.000us         0.00%       0.000us       0.000us       6.298ms         3.68%       6.298ms     572.552us            11  \n                                          ProfilerStep*         0.01%      21.566ms        99.98%      201.438s       67.146s       0.000us         0.00%     100.707ms      33.569ms             3  \nvoi

In [ ]:
os.cpu_count()

12

# Task
The user has approved the current state of the notebook. I will now modify the `SolarWindDataset` class to handle `s3fs` and `zarr` initialization per worker process, which is necessary for `num_workers` > 0 in `DataLoader` to avoid `asyncio` errors. After modifying the dataset, I will re-run the `DataLoader` setup, model initialization, and the training loop.

## Modify SolarWindDataset for Multiprocessing

### Subtask:
Adjust the `SolarWindDataset` class to ensure that `s3fs.S3FileSystem` and `zarr.open` calls are performed within a method (`_initialize_zarr`) that is called once per worker process (e.g., from `__getitem__`). This ensures each worker independently initializes its own S3 client and Zarr store, preventing `asyncio` conflicts. This cell will also include the DataLoader setup with `num_workers` set to `os.cpu_count()`, model initialization, and the training loop.


**Reasoning**:
The subtask requires modifying the `SolarWindDataset` class for multiprocessing compatibility, then re-initializing the data, model, and training loop. This involves creating a new `_initialize_zarr` method within the dataset class, updating its `__init__` and `__getitem__` methods, and subsequently re-running the data preparation and training process.



**Reasoning**:
The previous code block failed because 'df_aligned' was not defined. To fix this, I need to include the necessary steps to load data, define 'df_plasma' and 'data', and then create 'df_aligned' before performing the train-test split and training the model.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import os
import numpy as np
import pandas as pd
import s3fs
import zarr
from tqdm.auto import tqdm

AWS_ZARR_ROOT = (
    "s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/2018"
)
# Re-define functions for S3/Zarr connection and loading
def s3_connection(path_to_zarr: str) -> s3fs.S3Map:
    """
    Create an S3 connection for zarr access
    """
    s3 = s3fs.S3FileSystem(anon=True)  # anon=True for anonymous access (public bucket)
    return s3fs.S3Map(root=path_to_zarr, s3=s3)

def load_single_aws_zarr(
    path_to_zarr: str,
    cache_max_single_size: int | None = None,
) -> zarr.Array | zarr.Group:
    """
    Load zarr from S3 using the new zarr v3 API.
    Compatible with Python 3. 12+
    """
    store = s3_connection(path_to_zarr)
    return zarr.open(
        store=store,
        mode="r",
    )

# Load the dataset root and specific array
root = load_single_aws_zarr(
    path_to_zarr=AWS_ZARR_ROOT,
)
data = root["193A"]

# Load and preprocess df_plasma for 2018 (corrected from previous attempt)
df_plasma = pd.read_csv('solarwind2018.lst.txt',
                         sep=r'\s+',
                         header=None,
                         names=['year', 'day', 'hour', 'plasma_speed'])

# 2. Convert Year + Day + Hour into a single DatetimeIndex
# %j is the format code for Day of the Year (1-366)
df_plasma['time'] = pd.to_datetime(
    df_plasma['year'].astype(str) + '-' + df_plasma['day'].astype(str) + ' ' + df_plasma['hour'].astype(str) + ':00:00',
    format='%Y-%j %H:%M:%S'
)

# 3. Set the time as the index
df_plasma.set_index('time', inplace=True)

# 4. Clean the data
# OMNI files use 999.9 or 9999.0 for missing data.
# We replace it with NaN so it doesn't ruin the CNN training.
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].replace(9999.0, np.nan)

# 5. Interpolate missing values
# This fills small gaps in the 1-hour cadence so you have continuous targets for ML
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].interpolate(method='linear')

# 6. Keep only the necessary column
df_plasma = df_plasma[['plasma_speed']]

# Create df_aligned using the 4-day lag approach (from cell cVZs1Y3CNuMQ)
LAG_TIME = pd.Timedelta(days=4)

zarr_times_raw = data.attrs['DATE-OBS']
zarr_times_unsorted = pd.to_datetime(list(zarr_times_raw))
sort_order = np.argsort(zarr_times_unsorted)
zarr_times_sorted = zarr_times_unsorted[sort_order]

df_plasma['lookup_time'] = df_plasma.index - LAG_TIME

def get_nearest_sorted_indices(target_times, source_times_sorted):
    indices = np.searchsorted(source_times_sorted, target_times)
    indices = np.clip(indices, 1, len(source_times_sorted) - 1)
    left = source_times_sorted[indices - 1]
    right = source_times_sorted[indices]
    use_left = (target_times - left) < (right - target_times)
    return np.where(use_left, indices - 1, indices)

sorted_match_indices = get_nearest_sorted_indices(df_plasma['lookup_time'], zarr_times_sorted)
df_plasma['zarr_idx'] = sort_order[sorted_match_indices]

matched_timestamps = zarr_times_unsorted[df_plasma['zarr_idx'].values]
time_diffs = np.abs(matched_timestamps - df_plasma['lookup_time'])

df_aligned = df_plasma[time_diffs < pd.Timedelta(minutes=15)].copy()

# 1. Modified SolarWindDataset class for multiprocessing
class SolarWindDataset:
    def __init__(self, aligned_df, zarr_root):
        self.df = aligned_df
        self.zarr_root = zarr_root
        self.images = None  # Initialize images as None

    def _initialize_zarr(self):
        if self.images is None:
            # Create S3 file system within each worker process
            s3 = s3fs.S3FileSystem(anon=True)
            zarr_store = s3fs.S3Map(root=self.zarr_root, s3=s3)
            self.images = zarr.open(zarr_store, mode='r')["193A"]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        self._initialize_zarr() # Initialize zarr store per worker

        row = self.df.iloc[idx]
        # Pull image from Zarr using our saved index
        x = self.images[int(row['zarr_idx']), :, :]
        y = row['plasma_speed']

        # ML Preprocessing: Normalize and add channel dimension (1, 512, 512)
        x = np.log1p(np.maximum(x, 0))
        return x[None, :, :], y

# CNN Architecture
class SimpleSolarCNN(nn.Module):
    def __init__(self):
        super(SimpleSolarCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=9, stride=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.pool3 = nn.MaxPool2d(3, 1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=2, stride=2)

        self.fc1 = nn.Linear(8 * 8 * 128, 4096)
        self.fc2 = nn.Linear(4096, 1)

        self.dropout_fc = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool3((F.relu(self.conv3(x))))
        x = x.view(x.size(0), -1) # Flatten
        x = self.dropout_fc(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 2. Re-execute train_test_split
train_df, val_df = train_test_split(
    df_aligned,
    test_size=0.2,      # 80% train / 20% val
    random_state=322,
    shuffle=True
)

# 3. Re-scale Plasma Speed
scaler = MinMaxScaler()
train_df['plasma_speed'] = scaler.fit_transform(train_df[['plasma_speed']])
val_df['plasma_speed'] = scaler.transform(val_df[['plasma_speed']])

# 4. Re-initialize DataLoaders with modified SolarWindDataset
train_dataset = SolarWindDataset(train_df, AWS_ZARR_ROOT)
val_dataset   = SolarWindDataset(val_df, AWS_ZARR_ROOT)

# 5. Set up DataLoaders with num_workers = os.cpu_count()
num_workers = os.cpu_count()
print(f"Using {num_workers} workers for DataLoader.")
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

# 6. Re-initialize Model & Training Tools
model = SimpleSolarCNN()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Starting trial with {len(train_df)} training samples and {len(val_df)} validation samples using {device}...")

# 7. Execute Training Loop
num_epochs = 5
best_val_loss = float('inf')

for epoch in range(num_epochs):

    # ---------- TRAIN ----------
    model.train()
    train_loss = 0.0

    train_loop = tqdm(train_loader, leave=True)
    train_loop.set_description(f"Epoch [{epoch+1}/{num_epochs}] TRAIN")

    for images, speeds in train_loop:
        images = images.float().to(device)
        speeds = speeds.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, speeds)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    train_loss /= len(train_loader)

    # ---------- VALIDATION ----------
    model.eval()
    val_loss = 0.0

    val_loop = tqdm(val_loader, leave=False)
    val_loop.set_description(f"Epoch [{epoch+1}/{num_epochs}] VAL")

    with torch.no_grad():
        for images, speeds in val_loop:
            images = images.float().to(device)
            speeds = speeds.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, speeds)
            val_loss += loss.item()
            val_loop.set_postfix(loss=loss.item())

    val_loss /= len(val_loader)

    # ---------- LOG ----------
    print(f"Epoch {epoch+1}: Train {train_loss:.2f} | Val {val_loss:.2f}")

    # ---------- CHECKPOINT ----------
    ckpt = {
        'epoch': epoch,
        'model': model.state_dict(),
        'optim': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
    }

    torch.save(ckpt, "last_ckpt.pt")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        print("✅ Best model updated")

Using 12 workers for DataLoader.
Starting trial with 6793 training samples and 1699 validation samples using cuda...


  0%|          | 0/54 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import os
import numpy as np
import pandas as pd
import s3fs
import zarr
from tqdm.auto import tqdm

YEARS = [2013, 2014, 2015, 2017]

# Re-define functions for S3/Zarr connection and loading
def s3_connection(path_to_zarr: str) -> s3fs.S3Map:
    """
    Create an S3 connection for zarr access
    """
    s3 = s3fs.S3FileSystem(anon=True)  # anon=True for anonymous access (public bucket)
    return s3fs.S3Map(root=path_to_zarr, s3=s3)

def load_single_aws_zarr(
    path_to_zarr: str,
    cache_max_single_size: int | None = None,
) -> zarr.Array | zarr.Group:
    """
    Load zarr from S3 using the new zarr v3 API.
    Compatible with Python 3.12+
    """
    store = s3_connection(path_to_zarr)
    return zarr.open(
        store=store,
        mode="r",
    )

# Load data from all years
print("Loading Zarr data from S3...")
all_zarr_data = {}
all_zarr_roots = {}

for year in YEARS:
    zarr_path = f"s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/{year}"
    print(f"  Loading {year}...")
    root = load_single_aws_zarr(path_to_zarr=zarr_path)
    all_zarr_roots[year] = root
    all_zarr_data[year] = root["193A"]

print(f"Loaded {len(YEARS)} years of Zarr data")

# Load and preprocess df_plasma
print("\nLoading plasma data...")
df_plasma1 = pd.read_csv('omni2_sw131415.lst.txt',
                         sep=r'\s+',
                         header=None,
                         names=['year', 'day', 'hour', 'plasma_speed'])

df_plasma2 = pd.read_csv('omni2_2017.lst.txt',
                         sep=r'\s+',
                         header=None,
                         names=['year', 'day', 'hour', 'plasma_speed'])

df_plasma = pd.concat([df_plasma1, df_plasma2], ignore_index=True)

# 2. Convert Year + Day + Hour into a single DatetimeIndex
df_plasma['time'] = pd.to_datetime(
    df_plasma['year'].astype(str) + '-' + df_plasma['day'].astype(str) + ' ' + df_plasma['hour'].astype(str) + ':00:00',
    format='%Y-%j %H:%M:%S'
)

# 3. Set the time as the index
df_plasma.set_index('time', inplace=True)

# 4. Clean the data
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].replace(9999.0, np.nan)

# 5. Interpolate missing values
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].interpolate(method='linear')

# 6. Keep only the necessary column
df_plasma = df_plasma[['plasma_speed', 'year']]  # Keep year for filtering later

print(f"Loaded plasma data: {len(df_plasma)} records from {df_plasma.index.min()} to {df_plasma.index.max()}")

Loading Zarr data from S3...
  Loading 2013...
  Loading 2014...
  Loading 2015...
  Loading 2017...
Loaded 4 years of Zarr data

Loading plasma data...
Loaded plasma data: 35040 records from 2013-01-01 00:00:00 to 2017-12-31 23:00:00


In [ ]:
def get_nearest_sorted_indices(target_times, source_times_sorted):
    indices = np.searchsorted(source_times_sorted, target_times)
    indices = np.clip(indices, 1, len(source_times_sorted) - 1)

    left = source_times_sorted[indices - 1]
    right = source_times_sorted[indices]
    use_left = (target_times - left) < (right - target_times)

    return np.where(use_left, indices - 1, indices)

AU_KM = 149_597_870.7
all_aligned_dfs = []

for year in YEARS:
    print(f"\nProcessing year {year}...")

    # Filter plasma data for this year
    df_plasma_year = df_plasma[df_plasma['year'] == year].copy()

    # Get Zarr data for this year
    data_year = all_zarr_data[year]

    # Get and sort timestamps
    zarr_times_raw = data_year.attrs['DATE-OBS']
    zarr_times_unsorted = pd.to_datetime(list(zarr_times_raw))

    # Create a sorting index to keep track of where the data actually lives
    sort_order = np.argsort(zarr_times_unsorted)
    zarr_times_sorted = zarr_times_unsorted[sort_order]

    # Calculate time-of-flight based on plasma speed
    tof_seconds = AU_KM / df_plasma_year['plasma_speed']
    df_plasma_year['tof'] = pd.to_timedelta(tof_seconds, unit='s')
    df_plasma_year['lookup_time'] = df_plasma_year.index - df_plasma_year['tof']

    # Match plasma data to sorted timestamps
    sorted_match_indices = get_nearest_sorted_indices(
        df_plasma_year['lookup_time'].values,
        zarr_times_sorted
    )

    df_plasma_year['zarr_idx'] = sort_order[sorted_match_indices]

    # Filter with 15-minute tolerance
    matched_timestamps = zarr_times_unsorted[df_plasma_year['zarr_idx'].values]
    time_diffs = np.abs(matched_timestamps - df_plasma_year['lookup_time'])

    df_aligned_year = df_plasma_year[time_diffs < pd.Timedelta(minutes=15)].copy()
    df_aligned_year['year'] = year  # Keep track of which year each sample is from

    all_aligned_dfs.append(df_aligned_year)

    # Show TOF statistics for this year
    tof_days = df_aligned_year['tof'].dt.total_seconds() / 86400
    print(f"  Matched {len(df_aligned_year)} samples for {year}")
    print(f"  TOF (days) - Mean: {tof_days.mean():.2f}, Min: {tof_days.min():.2f}, Max: {tof_days.max():.2f}")

# Combine all years
df_aligned = pd.concat(all_aligned_dfs, ignore_index=False)
df_aligned = df_aligned.sort_index()
a
print(f"\n{'='*60}")
print(f"Dataset Ready: {len(df_aligned)} image-speed pairs.")
print(f"Date range: {df_aligned.index.min()} to {df_aligned.index.max()}")
print(f"\nOverall TOF Statistics (days):")
print((df_aligned['tof'].dt.total_seconds() / 86400).describe())


Processing year 2013...
  Matched 8236 samples for 2013
  TOF (days) - Mean: 4.54, Min: 2.21, Max: 6.90

Processing year 2014...
  Matched 7492 samples for 2014
  TOF (days) - Mean: 4.51, Min: 1.97, Max: 6.76

Processing year 2015...
  Matched 8269 samples for 2015
  TOF (days) - Mean: 4.15, Min: 2.16, Max: 6.69

Processing year 2017...
  Matched 8455 samples for 2017
  TOF (days) - Mean: 4.03, Min: 2.12, Max: 6.56

Dataset Ready: 32452 image-speed pairs.
Date range: 2013-01-06 11:00:00 to 2017-12-31 23:00:00

Overall TOF Statistics (days):
count    32452.000000
mean         4.299827
std          0.888105
min          1.972047
25%          3.660585
50%          4.328642
75%          4.947020
max          6.898234
Name: tof, dtype: float64


In [ ]:
class SolarWindDataset:
    def __init__(self, aligned_df, zarr_data_dict):
        """
        aligned_df: DataFrame with 'zarr_idx' and 'year' columns
        zarr_data_dict: Dictionary mapping year -> zarr array
        """
        self.df = aligned_df
        self.zarr_data_dict = zarr_data_dict
        self.images = {}  # Dictionary to store zarr connections per year

    def _initialize_zarr(self, year):
        if year not in self.images:
            # Create S3 file system within each worker process
            s3 = s3fs.S3FileSystem(anon=True)
            zarr_path = f"s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/{year}"
            zarr_store = s3fs.S3Map(root=zarr_path, s3=s3)
            self.images[year] = zarr.open(zarr_store, mode='r')["193A"]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        year = int(row['year'])

        # Initialize zarr store for this year if needed
        self._initialize_zarr(year)

        # Pull image from Zarr using our saved index
        x = self.images[year][int(row['zarr_idx']), :, :]
        y = row['plasma_speed']

        # Return raw data with channel dimension, preprocessing will be done on GPU
        return x[None, :, :], y

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Re-execute train_test_split
train_df, val_df = train_test_split(
    df_aligned,
    test_size=0.2,
    random_state=322,
    shuffle=True
)

# Re-scale Plasma Speed
scaler = MinMaxScaler()
train_df['plasma_speed'] = scaler.fit_transform(train_df[['plasma_speed']])
val_df['plasma_speed'] = scaler.transform(val_df[['plasma_speed']])

# Initialize datasets with the dictionary of zarr data
train_dataset = SolarWindDataset(train_df, all_zarr_data)
val_dataset = SolarWindDataset(val_df, all_zarr_data)

# DataLoaders
num_workers = os.cpu_count()
print(f"Using {num_workers} workers for DataLoader.")
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

class SimpleSolarCNN(nn.Module):
    def __init__(self):
        super(SimpleSolarCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=9, stride=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.pool3 = nn.MaxPool2d(3, 1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=2, stride=2)

        self.fc1 = nn.Linear(8 * 8 * 128, 4096)
        self.fc2 = nn.Linear(4096, 1)

        self.dropout_fc = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool3((F.relu(self.conv3(x))))
        x = x.view(x.size(0), -1) # Flatten
        x = self.dropout_fc(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

Using 12 workers for DataLoader.


In [ ]:
# Initialize model, loss, and optimizer
model = SimpleSolarCNN()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Starting trial with {len(train_df)} training samples and {len(val_df)} validation samples using {device}...")

# Check GPU memory after model initialization
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Model size: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

# Execute Training Loop with GPU preprocessing
num_epochs = 5
best_val_loss = float('inf')

for epoch in range(num_epochs):

    # ---------- TRAIN ----------
    model.train()
    train_loss = 0.0

    train_loop = tqdm(train_loader, leave=True)
    train_loop.set_description(f"Epoch [{epoch+1}/{num_epochs}] TRAIN")

    for images, speeds in train_loop:
        # Move to GPU first, THEN apply log1p
        images = images.float().to(device)
        images = torch.log1p(torch.clamp(images, min=0))  # GPU operation
        speeds = speeds.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, speeds)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    train_loss /= len(train_loader)

    # ---------- VALIDATION ----------
    model.eval()
    val_loss = 0.0

    val_loop = tqdm(val_loader, leave=False)
    val_loop.set_description(f"Epoch [{epoch+1}/{num_epochs}] VAL")

    with torch.no_grad():
        for images, speeds in val_loop:
            # Move to GPU first, THEN apply log1p
            images = images.float().to(device)
            images = torch.log1p(torch.clamp(images, min=0))  # GPU operation
            speeds = speeds.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, speeds)
            val_loss += loss.item()
            val_loop.set_postfix(loss=loss.item())

    val_loss /= len(val_loader)

    # ---------- LOG ----------
    print(f"Epoch {epoch+1}: Train {train_loss:.6f} | Val {val_loss:.6f}")  # Changed to 6 decimals

    # ---------- CHECKPOINT ----------
    ckpt = {
        'epoch': epoch,
        'model': model.state_dict(),
        'optim': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'scaler': scaler,  # Save scaler for inference
    }

    torch.save(ckpt, "last_ckpt.pt")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(ckpt, "best_model.pt")  # Save full checkpoint, not just model
        print(f"✅ Best model updated (val_loss: {val_loss:.6f})")

print(f"\n{'='*60}")
print(f"Training completed!")
print(f"Best validation loss: {best_val_loss:.6f}")

Starting trial with 25961 training samples and 6491 validation samples using cuda...
GPU: NVIDIA A100-SXM4-80GB
Model size: 128.20 MB


  0%|          | 0/203 [00:00<?, ?it/s]

  0%|          | 0/51 [00:00<?, ?it/s]

Epoch 1: Train 1.001146 | Val 0.023840
✅ Best model updated (val_loss: 0.023840)


  0%|          | 0/203 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea9ccd35ee0>
Exception ignored in: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea9ccd35ee0>
self._shutdown_workers()Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ea9ccd35ee0>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    self._shutdown_workers()    
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    self._shutdown_workers()

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import s3fs
import zarr
from tqdm.auto import tqdm
from sklearn.preprocessing import MinMaxScaler

# ============================================
# 1. Load the trained model
# ============================================

class SimpleSolarCNN(nn.Module):
    def __init__(self):
        super(SimpleSolarCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=9, stride=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.pool3 = nn.MaxPool2d(3, 1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=2, stride=2)

        self.fc1 = nn.Linear(8 * 8 * 128, 4096)
        self.fc2 = nn.Linear(4096, 1)

        self.dropout_fc = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool3((F.relu(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout_fc(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleSolarCNN()
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.to(device)
model.eval()

print(f"✅ Model loaded on {device}")

# ============================================
# 2. Load the scaler (reconstruct from training data)
# ============================================

# Re-load plasma data
df_plasma = pd.read_csv('solarwind2018.lst.txt',
                         sep=r'\s+',
                         header=None,
                         names=['year', 'day', 'hour', 'plasma_speed'])

df_plasma['time'] = pd.to_datetime(
    df_plasma['year'].astype(str) + '-' + df_plasma['day'].astype(str) + ' ' + df_plasma['hour'].astype(str) + ':00:00',
    format='%Y-%j %H:%M:%S'
)
df_plasma.set_index('time', inplace=True)
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].replace(9999.0, np.nan)
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].interpolate(method='linear')
df_plasma = df_plasma[['plasma_speed']].copy()

# Recreate the scaler from the full dataset
scaler = MinMaxScaler()
scaler.fit(df_plasma[['plasma_speed']])

print(f"✅ Scaler fitted on full plasma dataset")

# ============================================
# 3. Load Zarr data
# ============================================

AWS_ZARR_ROOT = (
    "s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/2018"
)

def s3_connection(path_to_zarr: str) -> s3fs.S3Map:
    s3 = s3fs.S3FileSystem(anon=True)
    return s3fs.S3Map(root=path_to_zarr, s3=s3)

def load_single_aws_zarr(path_to_zarr: str) -> zarr.Array | zarr.Group:
    store = s3_connection(path_to_zarr)
    return zarr.open(store=store, mode="r")

root = load_single_aws_zarr(path_to_zarr=AWS_ZARR_ROOT)
data = root["193A"]

zarr_times_raw = data.attrs['DATE-OBS']
zarr_times = pd.to_datetime(list(zarr_times_raw))

print(f"✅ Loaded {len(zarr_times)} images from Zarr")

# ============================================
# 4. Inference with Predicted ToF -> Ground Truth Lookup
# ============================================

AU_KM = 149_597_870.7

def get_nearest_index(target_time, time_array):
    """Find the nearest index in time_array to target_time"""
    idx = np.searchsorted(time_array, target_time)
    idx = np.clip(idx, 1, len(time_array) - 1)

    left = time_array[idx - 1]
    right = time_array[idx]
    use_left = (target_time - left) < (right - target_time)

    return idx - 1 if use_left else idx

def preprocess_image(image):
    """Preprocess image for model input"""
    x = np.log1p(np.maximum(image, 0))
    x = torch.from_numpy(x[None, None, :, :]).float()  # Add batch and channel dims
    return x

def get_ground_truth_at_time(target_time, plasma_df):
    """Get ground truth plasma speed at a specific time via interpolation"""
    # Find nearest times in plasma dataframe
    try:
        # Use reindex with nearest method for fast lookup
        return plasma_df.reindex([target_time], method='nearest', tolerance=pd.Timedelta(hours=1))['plasma_speed'].iloc[0]
    except:
        return np.nan

results = []

print("\n🚀 Starting inference with predicted ToF...")

# Iterate through each plasma observation at Earth
for obs_time, row in tqdm(df_plasma.iterrows(), total=len(df_plasma), desc="Testing"):

    obs_ground_truth_speed = row['plasma_speed']

    # Skip if ground truth is NaN
    if pd.isna(obs_ground_truth_speed):
        continue

    # === STEP 1: Find the image at observation time ===
    zarr_idx = get_nearest_index(obs_time, zarr_times)

    # Check if the match is within tolerance (15 minutes)
    time_diff = abs(zarr_times[zarr_idx] - obs_time)
    if time_diff > pd.Timedelta(minutes=15):
        continue

    # === STEP 2: Make prediction ===
    image = data[zarr_idx, :, :]
    image_tensor = preprocess_image(image).to(device)

    with torch.no_grad():
        pred_normalized = model(image_tensor).cpu().item()

    # Inverse transform to get actual speed
    pred_speed = scaler.inverse_transform([[pred_normalized]])[0, 0]

    # === STEP 3: Calculate ToF using PREDICTED speed ===
    pred_tof_seconds = AU_KM / pred_speed
    pred_tof = pd.Timedelta(seconds=pred_tof_seconds)

    # Calculate when this solar wind should have left the Sun
    pred_lookup_time = obs_time - pred_tof

    # === STEP 4: Find GROUND TRUTH at the predicted lookup time ===
    # We need to find what the actual solar wind speed was at pred_lookup_time
    # and then calculate when THAT wind would arrive at Earth

    # Get ground truth speed at the predicted departure time
    gt_speed_at_pred_time = get_ground_truth_at_time(pred_lookup_time, df_plasma)

    if pd.isna(gt_speed_at_pred_time):
        continue

    # Calculate when this ground truth wind would actually arrive at Earth
    gt_tof_seconds = AU_KM / gt_speed_at_pred_time
    gt_tof = pd.Timedelta(seconds=gt_tof_seconds)
    gt_arrival_time = pred_lookup_time + gt_tof

    # === STEP 5: Get the ground truth speed at the predicted arrival time ===
    # This is the ground truth we compare against
    final_ground_truth_speed = get_ground_truth_at_time(gt_arrival_time, df_plasma)

    if pd.isna(final_ground_truth_speed):
        continue

    # === STEP 6: Store results ===
    results.append({
        'obs_time': obs_time,
        'image_time': zarr_times[zarr_idx],
        'pred_speed': pred_speed,
        'pred_tof_days': pred_tof_seconds / 86400,
        'pred_lookup_time': pred_lookup_time,
        'gt_speed_at_pred_time': gt_speed_at_pred_time,
        'gt_tof_days': gt_tof_seconds / 86400,
        'gt_arrival_time': gt_arrival_time,
        'ground_truth_speed': final_ground_truth_speed,
        'zarr_idx': zarr_idx,
        'time_diff_minutes': time_diff.total_seconds() / 60
    })

# ============================================
# 5. Convert to DataFrame and Analyze
# ============================================

df_results = pd.DataFrame(results)

print(f"\n✅ Testing complete! {len(df_results)} valid predictions made.")

# Calculate metrics
mae = np.mean(np.abs(df_results['pred_speed'] - df_results['ground_truth_speed']))
rmse = np.sqrt(np.mean((df_results['pred_speed'] - df_results['ground_truth_speed'])**2))
mape = np.mean(np.abs((df_results['pred_speed'] - df_results['ground_truth_speed']) / df_results['ground_truth_speed'])) * 100

print("\n📊 Evaluation Metrics:")
print(f"MAE:  {mae:.2f} km/s")
print(f"RMSE: {rmse:.2f} km/s")
print(f"MAPE: {mape:.2f}%")

print("\n📈 Speed Statistics:")
print(df_results[['pred_speed', 'ground_truth_speed', 'gt_speed_at_pred_time']].describe())

print("\n⏱️  ToF Statistics:")
print(f"Mean predicted ToF: {df_results['pred_tof_days'].mean():.2f} days")
print(f"Std predicted ToF:  {df_results['pred_tof_days'].std():.2f} days")
print(f"Mean ground truth ToF: {df_results['gt_tof_days'].mean():.2f} days")
print(f"Std ground truth ToF:  {df_results['gt_tof_days'].std():.2f} days")

# Save results
df_results.to_csv('test_results_with_predicted_tof.csv', index=False)
print("\n💾 Results saved to 'test_results_with_predicted_tof.csv'")

# Additional analysis: Compare ToF predictions
tof_error = np.abs(df_results['pred_tof_days'] - df_results['gt_tof_days'])
print(f"\n⏱️  ToF Prediction Error:")
print(f"Mean ToF error: {tof_error.mean():.2f} days")
print(f"Median ToF error: {tof_error.median():.2f} days")

✅ Model loaded on cuda
✅ Scaler fitted on full plasma dataset
✅ Loaded 84881 images from Zarr

🚀 Starting inference with predicted ToF...


Testing:   0%|          | 0/8760 [00:00<?, ?it/s]


✅ Testing complete! 5491 valid predictions made.

📊 Evaluation Metrics:
MAE:  71.21 km/s
RMSE: 90.37 km/s
MAPE: 16.88%

📈 Speed Statistics:
        pred_speed  ground_truth_speed  gt_speed_at_pred_time
count  5491.000000         5491.000000            5491.000000
mean    419.641096          423.380259             410.501184
std      35.856619           86.063841              84.344923
min     332.741904          277.000000             279.000000
25%     392.126100          356.000000             347.000000
50%     415.168851          407.000000             390.000000
75%     441.530504          477.000000             457.000000
max     554.455162          695.000000             701.000000

⏱️  ToF Statistics:
Mean predicted ToF: 4.16 days
Std predicted ToF:  0.34 days
Mean ground truth ToF: 4.38 days
Std ground truth ToF:  0.81 days

💾 Results saved to 'test_results_with_predicted_tof.csv'

⏱️  ToF Prediction Error:
Mean ToF error: 0.75 days
Median ToF error: 0.69 days


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import s3fs
import zarr
from tqdm.auto import tqdm
from sklearn.preprocessing import MinMaxScaler

# ============================================
# 1. Load the trained model
# ============================================

class SimpleSolarCNN(nn.Module):
    def __init__(self):
        super(SimpleSolarCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=9, stride=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.pool3 = nn.MaxPool2d(3, 1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=2, stride=2)

        self.fc1 = nn.Linear(8 * 8 * 128, 4096)
        self.fc2 = nn.Linear(4096, 1)

        self.dropout_fc = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool3((F.relu(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout_fc(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleSolarCNN()
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.to(device)
model.eval()

print(f"✅ Model loaded on {device}")

# ============================================
# 2. Load the scaler (reconstruct from training data)
# ============================================

# Re-load plasma data
df_plasma = pd.read_csv('solarwind2018.lst.txt',
                         sep=r'\s+',
                         header=None,
                         names=['year', 'day', 'hour', 'plasma_speed'])

df_plasma['time'] = pd.to_datetime(
    df_plasma['year'].astype(str) + '-' + df_plasma['day'].astype(str) + ' ' + df_plasma['hour'].astype(str) + ':00:00',
    format='%Y-%j %H:%M:%S'
)
df_plasma.set_index('time', inplace=True)
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].replace(9999.0, np.nan)
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].interpolate(method='linear')
df_plasma = df_plasma[['plasma_speed']].copy()

# Recreate the scaler from the full dataset
scaler = MinMaxScaler()
scaler.fit(df_plasma[['plasma_speed']])

print(f"✅ Scaler fitted on full plasma dataset")

# ============================================
# 3. Load Zarr data
# ============================================

AWS_ZARR_ROOT = (
    "s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/2018"
)

def s3_connection(path_to_zarr: str) -> s3fs.S3Map:
    s3 = s3fs.S3FileSystem(anon=True)
    return s3fs.S3Map(root=path_to_zarr, s3=s3)

def load_single_aws_zarr(path_to_zarr: str) -> zarr.Array | zarr.Group:
    store = s3_connection(path_to_zarr)
    return zarr.open(store=store, mode="r")

root = load_single_aws_zarr(path_to_zarr=AWS_ZARR_ROOT)
data = root["193A"]

zarr_times_raw = data.attrs['DATE-OBS']
zarr_times = pd.to_datetime(list(zarr_times_raw))

print(f"✅ Loaded {len(zarr_times)} images from Zarr")

# ============================================
# 4. Inference with FIXED 4-day Time Delta
# ============================================

FIXED_TOF_DAYS = 4
FIXED_TOF = pd.Timedelta(days=FIXED_TOF_DAYS)

def get_nearest_index(target_time, time_array):
    """Find the nearest index in time_array to target_time"""
    idx = np.searchsorted(time_array, target_time)
    idx = np.clip(idx, 1, len(time_array) - 1)

    left = time_array[idx - 1]
    right = time_array[idx]
    use_left = (target_time - left) < (right - target_time)

    return idx - 1 if use_left else idx

def preprocess_image(image):
    """Preprocess image for model input"""
    x = np.log1p(np.maximum(image, 0))
    x = torch.from_numpy(x[None, None, :, :]).float()  # Add batch and channel dims
    return x

results_fixed = []

print(f"\n🚀 Starting inference with FIXED {FIXED_TOF_DAYS}-day time delta...")

# Iterate through each plasma observation at Earth
for obs_time, row in tqdm(df_plasma.iterrows(), total=len(df_plasma), desc="Testing (Fixed ToF)"):

    ground_truth_speed = row['plasma_speed']

    # Skip if ground truth is NaN
    if pd.isna(ground_truth_speed):
        continue

    # === STEP 1: Calculate lookup time using FIXED 4-day delta ===
    lookup_time = obs_time - FIXED_TOF

    # === STEP 2: Find the image at lookup time ===
    zarr_idx = get_nearest_index(lookup_time, zarr_times)

    # Check if the match is within tolerance (15 minutes)
    time_diff = abs(zarr_times[zarr_idx] - lookup_time)
    if time_diff > pd.Timedelta(minutes=15):
        continue

    # === STEP 3: Make prediction ===
    image = data[zarr_idx, :, :]
    image_tensor = preprocess_image(image).to(device)

    with torch.no_grad():
        pred_normalized = model(image_tensor).cpu().item()

    # Inverse transform to get actual speed
    pred_speed = scaler.inverse_transform([[pred_normalized]])[0, 0]

    # === STEP 4: Store results ===
    results_fixed.append({
        'obs_time': obs_time,
        'lookup_time': lookup_time,
        'image_time': zarr_times[zarr_idx],
        'pred_speed': pred_speed,
        'ground_truth_speed': ground_truth_speed,
        'fixed_tof_days': FIXED_TOF_DAYS,
        'zarr_idx': zarr_idx,
        'time_diff_minutes': time_diff.total_seconds() / 60
    })

    # Periodic memory cleanup
    if len(results_fixed) % 1000 == 0:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# ============================================
# 5. Convert to DataFrame and Analyze
# ============================================

df_results_fixed = pd.DataFrame(results_fixed)

print(f"\n✅ Testing complete! {len(df_results_fixed)} valid predictions made.")

# Calculate metrics
mae_fixed = np.mean(np.abs(df_results_fixed['pred_speed'] - df_results_fixed['ground_truth_speed']))
rmse_fixed = np.sqrt(np.mean((df_results_fixed['pred_speed'] - df_results_fixed['ground_truth_speed'])**2))
mape_fixed = np.mean(np.abs((df_results_fixed['pred_speed'] - df_results_fixed['ground_truth_speed']) / df_results_fixed['ground_truth_speed'])) * 100

print("\n" + "="*60)
print(f"📊 Evaluation Metrics (Fixed {FIXED_TOF_DAYS}-day ToF)")
print("="*60)
print(f"MAE:  {mae_fixed:.2f} km/s")
print(f"RMSE: {rmse_fixed:.2f} km/s")
print(f"MAPE: {mape_fixed:.2f}%")

print("\n📈 Speed Statistics:")
print(df_results_fixed[['pred_speed', 'ground_truth_speed']].describe())

# Calculate correlation
correlation = df_results_fixed[['pred_speed', 'ground_truth_speed']].corr().iloc[0, 1]
print(f"\n📊 Correlation between predicted and ground truth: {correlation:.4f}")

# Save results
df_results_fixed.to_csv('test_results_fixed_4day_tof.csv', index=False)
print(f"\n💾 Results saved to 'test_results_fixed_4day_tof.csv'")

# Additional statistics
print("\n" + "="*60)
print("Additional Analysis")
print("="*60)

# Error distribution
errors = df_results_fixed['pred_speed'] - df_results_fixed['ground_truth_speed']
print(f"\nError Statistics:")
print(f"  Mean error (bias): {errors.mean():.2f} km/s")
print(f"  Std error: {errors.std():.2f} km/s")
print(f"  Median error: {errors.median():.2f} km/s")
print(f"  Min error: {errors.min():.2f} km/s")
print(f"  Max error: {errors.max():.2f} km/s")

# Percentage within certain error thresholds
within_50 = (np.abs(errors) <= 50).sum() / len(errors) * 100
within_100 = (np.abs(errors) <= 100).sum() / len(errors) * 100
within_150 = (np.abs(errors) <= 150).sum() / len(errors) * 100

print(f"\nPredictions within error threshold:")
print(f"  ±50 km/s:  {within_50:.1f}%")
print(f"  ±100 km/s: {within_100:.1f}%")
print(f"  ±150 km/s: {within_150:.1f}%")

print("\n✅ Fixed 4-day ToF evaluation complete!")

✅ Model loaded on cuda
✅ Scaler fitted on full plasma dataset
✅ Loaded 84881 images from Zarr

🚀 Starting inference with FIXED 4-day time delta...


Testing (Fixed ToF):   0%|          | 0/8760 [00:00<?, ?it/s]


✅ Testing complete! 5446 valid predictions made.

📊 Evaluation Metrics (Fixed 4-day ToF)
MAE:  54.58 km/s
RMSE: 67.37 km/s
MAPE: 13.39%

📈 Speed Statistics:
        pred_speed  ground_truth_speed
count  5446.000000         5446.000000
mean    419.716736          415.293977
std      36.014200           81.886327
min     332.741904          272.000000
25%     391.776905          350.000000
50%     415.360076          400.000000
75%     442.101867          470.000000
max     554.455162          688.000000

📊 Correlation between predicted and ground truth: 0.5903

💾 Results saved to 'test_results_fixed_4day_tof.csv'

Additional Analysis

Error Statistics:
  Mean error (bias): 4.42 km/s
  Std error: 67.23 km/s
  Median error: 15.88 km/s
  Min error: -224.87 km/s
  Max error: 169.44 km/s

Predictions within error threshold:
  ±50 km/s:  52.8%
  ±100 km/s: 86.3%
  ±150 km/s: 97.8%

✅ Fixed 4-day ToF evaluation complete!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import os
import numpy as np
import pandas as pd
import s3fs
import zarr
from tqdm.auto import tqdm

AWS_ZARR_ROOT = (
    "s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/2018"
)

def s3_connection(path_to_zarr: str) -> s3fs.S3Map:
    s3 = s3fs.S3FileSystem(anon=True)
    return s3fs.S3Map(root=path_to_zarr, s3=s3)

def load_single_aws_zarr(path_to_zarr: str) -> zarr.Array | zarr.Group:
    store = s3_connection(path_to_zarr)
    return zarr.open(store=store, mode="r")

# Load the dataset root and specific array
root = load_single_aws_zarr(path_to_zarr=AWS_ZARR_ROOT)
data = root["193A"]

# Load and preprocess df_plasma for 2018
df_plasma = pd.read_csv('solarwind2018.lst.txt',
                         sep=r'\s+',
                         header=None,
                         names=['year', 'day', 'hour', 'plasma_speed'])

df_plasma['time'] = pd.to_datetime(
    df_plasma['year'].astype(str) + '-' + df_plasma['day'].astype(str) + ' ' + df_plasma['hour'].astype(str) + ':00:00',
    format='%Y-%j %H:%M:%S'
)

df_plasma.set_index('time', inplace=True)
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].replace(9999.0, np.nan)
df_plasma['plasma_speed'] = df_plasma['plasma_speed'].interpolate(method='linear')
df_plasma = df_plasma[['plasma_speed']]

# Create df_aligned
zarr_times_raw = data.attrs['DATE-OBS']
zarr_times_unsorted = pd.to_datetime(list(zarr_times_raw))

sort_order = np.argsort(zarr_times_unsorted)
zarr_times_sorted = zarr_times_unsorted[sort_order]

def get_nearest_sorted_indices(target_times, source_times_sorted):
    indices = np.searchsorted(source_times_sorted, target_times)
    indices = np.clip(indices, 1, len(source_times_sorted) - 1)
    left = source_times_sorted[indices - 1]
    right = source_times_sorted[indices]
    use_left = (target_times - left) < (right - target_times)
    return np.where(use_left, indices - 1, indices)

AU_KM = 149_597_870.7
tof_seconds = AU_KM / df_plasma['plasma_speed']
df_plasma['tof'] = pd.to_timedelta(tof_seconds, unit='s')
df_plasma['lookup_time'] = df_plasma.index - df_plasma['tof']

sorted_match_indices = get_nearest_sorted_indices(
    df_plasma['lookup_time'].values,
    zarr_times_sorted
)

df_plasma['zarr_idx'] = sort_order[sorted_match_indices]

matched_timestamps = zarr_times_unsorted[df_plasma['zarr_idx'].values]
time_diffs = np.abs(matched_timestamps - df_plasma['lookup_time'])
df_aligned = df_plasma[time_diffs < pd.Timedelta(minutes=15)].copy()

print(f"Dataset Ready: {len(df_aligned)} image-speed pairs.")

# ============================================
# FIXED: Worker initialization function
# ============================================
def worker_init_fn(worker_id):
    """Initialize each worker with its own Zarr connection"""
    worker_info = torch.utils.data.get_worker_info()
    if worker_info is not None:
        dataset = worker_info.dataset
        # Each worker gets its own S3 connection
        s3 = s3fs.S3FileSystem(anon=True)
        zarr_store = s3fs.S3Map(root=dataset.zarr_root, s3=s3)
        dataset.images = zarr.open(zarr_store, mode='r')["193A"]

# ============================================
# FIXED: Dataset class
# ============================================
from torch.utils.data import Dataset

class SolarWindDataset(Dataset):
    def __init__(self, aligned_df, zarr_root, chunk_size=256):
        self.df = aligned_df.reset_index(drop=True)
        self.zarr_root = zarr_root
        self.images = None
        self.chunk_size = chunk_size
        self.cache = {}

    def __len__(self):
        return len(self.df)

    def _fetch_chunk(self, chunk_id):
        """Fetch a chunk of images at once for better S3 performance"""
        start_idx = chunk_id * self.chunk_size
        end_idx = min(start_idx + self.chunk_size, len(self.df))

        # Get all indices in this chunk
        indices = self.df.iloc[start_idx:end_idx]['zarr_idx'].values.astype(int)

        # Single S3 request for multiple images
        chunk_data = self.images.get_orthogonal_selection(
            (indices, slice(None), slice(None))
        )

        return chunk_data

    def __getitem__(self, idx):
        # Lazy init for workers
        if self.images is None:
            s3 = s3fs.S3FileSystem(anon=True)
            zarr_store = s3fs.S3Map(root=self.zarr_root, s3=s3)
            self.images = zarr.open(zarr_store, mode='r')["193A"]

        # Determine which chunk this index belongs to
        chunk_id = idx // self.chunk_size

        # Fetch chunk if not in cache (keep only 1 chunk in memory)
        if chunk_id not in self.cache:
            self.cache = {chunk_id: self._fetch_chunk(chunk_id)}

        # Get image from cached chunk
        local_idx = idx % self.chunk_size
        chunk_end = min((chunk_id + 1) * self.chunk_size, len(self.df))

        if local_idx >= len(self.cache[chunk_id]):
            local_idx = len(self.cache[chunk_id]) - 1

        x = self.cache[chunk_id][local_idx]

        # Get target
        row = self.df.iloc[idx]
        y = row['plasma_speed']

        # Preprocessing
        x = np.log1p(np.maximum(x, 0))
        return x[None, :, :].astype(np.float32), np.float32(y)

# Train/test split
train_df, val_df = train_test_split(
    df_aligned,
    test_size=0.2,
    random_state=322,
    shuffle=True
)

# Scale plasma speed
scaler = MinMaxScaler()
train_df = train_df.copy()  # Avoid SettingWithCopyWarning
val_df = val_df.copy()

train_df['plasma_speed'] = scaler.fit_transform(train_df[['plasma_speed']])
val_df['plasma_speed'] = scaler.transform(val_df[['plasma_speed']])

# Initialize datasets
train_dataset = SolarWindDataset(train_df, AWS_ZARR_ROOT)
val_dataset = SolarWindDataset(val_df, AWS_ZARR_ROOT)

# ============================================
# CRITICAL FIX: Limit number of workers
# ============================================
num_workers = os.cpu_count()
print(f"Using {num_workers} workers for DataLoader.")

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,  # Reuse workers
    worker_init_fn=worker_init_fn,  # Proper initialization
    prefetch_factor=2,  # Reduce prefetch buffer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    worker_init_fn=worker_init_fn,
    prefetch_factor=2,
)

# Initialize model
model = SimpleSolarCNN()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Starting trial with {len(train_df)} training samples and {len(val_df)} validation samples using {device}...")

# ============================================
# FIXED: Training loop with memory monitoring
# ============================================
import psutil

def get_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / 1024**3  # GB

num_epochs = 7
best_val_loss = float('inf')

print(f"Initial RAM usage: {get_memory_usage():.2f} GB")

try:
    for epoch in range(num_epochs):
        # ---------- TRAIN ----------
        model.train()
        train_loss = 0.0

        train_loop = tqdm(train_loader, leave=True)
        train_loop.set_description(f"Epoch [{epoch+1}/{num_epochs}] TRAIN")

        for images, speeds in train_loop:
            images = images.to(device, non_blocking=True)
            speeds = speeds.unsqueeze(1).to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, speeds)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_loop.set_postfix(loss=loss.item())

        train_loss /= len(train_loader)

        # ---------- VALIDATION ----------
        model.eval()
        val_loss = 0.0

        val_loop = tqdm(val_loader, leave=False)
        val_loop.set_description(f"Epoch [{epoch+1}/{num_epochs}] VAL")

        with torch.no_grad():
            for images, speeds in val_loop:
                images = images.to(device, non_blocking=True)
                speeds = speeds.unsqueeze(1).to(device, non_blocking=True)

                outputs = model(images)
                loss = criterion(outputs, speeds)
                val_loss += loss.item()
                val_loop.set_postfix(loss=loss.item())

        val_loss /= len(val_loader)

        # ---------- LOG ----------
        mem_usage = get_memory_usage()
        print(f"Epoch {epoch+1}: Train {train_loss:.4f} | Val {val_loss:.4f} | RAM {mem_usage:.2f} GB")

        # ---------- CHECKPOINT ----------
        ckpt = {
            'epoch': epoch,
            'model': model.state_dict(),
            'optim': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'scaler': scaler,  # Save scaler too
        }

        torch.save(ckpt, "last_ckpt.pt")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pt")
            print("✅ Best model updated")

        # Clear cache periodically
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

finally:
    # Cleanup
    print("\n🧹 Cleaning up...")
    del train_loader
    del val_loader
    import gc
    gc.collect()
    print(f"Final RAM usage: {get_memory_usage():.2f} GB")

Dataset Ready: 8417 image-speed pairs.
Using 12 workers for DataLoader.
Starting trial with 6733 training samples and 1684 validation samples using cuda...
Initial RAM usage: 3.51 GB


  0%|          | 0/53 [00:00<?, ?it/s]


🧹 Cleaning up...
Final RAM usage: 3.51 GB


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os
import numpy as np
import pandas as pd
import s3fs
import zarr
from tqdm.auto import tqdm
import time
import matplotlib.pyplot as plt

# ============================================
# Configuration
# ============================================
TEST_YEAR = 2017
CACHE_DIR = './cached_data'
DELAY_DAYS = 4  # Fixed 4-day delay
os.makedirs(CACHE_DIR, exist_ok=True)

# ============================================
# Function Definitions
# ============================================
def s3_connection(path_to_zarr: str) -> s3fs.S3Map:
    """Create S3 connection"""
    s3 = s3fs.S3FileSystem(anon=True)
    return s3fs.S3Map(root=path_to_zarr, s3=s3)

def load_single_aws_zarr(path_to_zarr: str) -> zarr.Array | zarr.Group:
    """Load zarr from S3"""
    store = s3_connection(path_to_zarr)
    return zarr.open(store=store, mode="r")

def get_nearest_sorted_indices(target_times, source_times_sorted):
    """Find nearest timestamp indices"""
    indices = np.searchsorted(source_times_sorted, target_times)
    indices = np.clip(indices, 1, len(source_times_sorted) - 1)
    left = source_times_sorted[indices - 1]
    right = source_times_sorted[indices]
    use_left = (target_times - left) < (right - target_times)
    return np.where(use_left, indices - 1, indices)

# ============================================
# Cache Images Function (Same Style as Training)
# ============================================
def cache_images_for_year(df_aligned, zarr_data, year, cache_dir=CACHE_DIR):
    """
    Cache images for a single year
    Same style as the multi-year training cache function
    """
    print("\n" + "="*60)
    print(f"CACHING IMAGES FOR {year}")
    print("="*60)

    cache_file = os.path.join(cache_dir, f'images_{year}.npy')
    index_mapping = {}

    # Check if cache already exists
    if os.path.exists(cache_file):
        print(f"\n✅ Cache already exists: {cache_file}")
        file_size_gb = os.path.getsize(cache_file) / 1024**3
        print(f"   Size: {file_size_gb:.2f} GB")

        # Build index mapping
        for local_idx, (row_idx, row) in enumerate(df_aligned.iterrows()):
            index_mapping[row_idx] = (year, local_idx)

        print(f"   Created index mapping for {len(index_mapping)} samples")
        print("="*60 + "\n")
        return index_mapping

    # Download images from S3
    print(f"\n⏬ Downloading images for {year}...")
    start_time = time.time()

    zarr_indices = df_aligned['zarr_idx'].values.astype(int)
    print(f"   Total images to download: {len(zarr_indices)}")

    # Download in chunks to avoid memory issues
    chunk_size = 1000
    all_images = []

    print(f"   Downloading in chunks of {chunk_size}...")
    for i in tqdm(range(0, len(zarr_indices), chunk_size), desc=f"   {year}"):
        chunk_idx = zarr_indices[i:i+chunk_size]

        try:
            chunk_images = zarr_data.get_orthogonal_selection(
                (chunk_idx, slice(None), slice(None))
            )
            all_images.append(chunk_images)
        except Exception as e:
            print(f"\n   ⚠️  Error downloading chunk {i//chunk_size + 1}: {e}")
            print(f"   Retrying...")
            time.sleep(2)
            chunk_images = zarr_data.get_orthogonal_selection(
                (chunk_idx, slice(None), slice(None))
            )
            all_images.append(chunk_images)

    # Concatenate all chunks
    print(f"\n   Concatenating {len(all_images)} chunks...")
    all_images = np.concatenate(all_images, axis=0)

    # Save to disk
    print(f"   Saving to {cache_file}...")
    np.save(cache_file, all_images)

    elapsed = time.time() - start_time
    file_size_gb = os.path.getsize(cache_file) / 1024**3

    print(f"\n   ✅ Saved {cache_file}")
    print(f"   Time: {elapsed:.1f}s ({elapsed/60:.1f} minutes)")
    print(f"   Size: {file_size_gb:.2f} GB")
    print(f"   Download speed: {file_size_gb/(elapsed/60):.2f} GB/min")

    # Build index mapping
    for local_idx, (row_idx, row) in enumerate(df_aligned.iterrows()):
        index_mapping[row_idx] = (year, local_idx)

    print(f"   Created index mapping for {len(index_mapping)} samples")

    print("\n" + "="*60)
    print("✅ CACHING COMPLETE!")
    print("="*60 + "\n")

    return index_mapping

# ============================================
# Dataset Class (Same as Training)
# ============================================
class CachedSolarWindDataset(Dataset):
    def __init__(self, aligned_df, index_mapping, year, cache_dir=CACHE_DIR):
        """
        aligned_df: DataFrame with 'zarr_idx' and 'year' columns
        index_mapping: Dictionary mapping DataFrame index -> (year, local_idx)
        year: Year of the data
        cache_dir: Directory containing cached .npy files
        """
        self.df = aligned_df.reset_index(drop=False)
        self.index_mapping = index_mapping
        self.year = year

        # Load memory-mapped file
        cache_file = os.path.join(cache_dir, f'images_{year}.npy')
        self.cached_images = np.load(cache_file, mmap_mode='r')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        original_index = row['time']

        year, local_idx = self.index_mapping[original_index]
        x = self.cached_images[local_idx]
        y = row['plasma_speed']

        return x[None, :, :].astype(np.float32), np.float32(y)

# ============================================
# Model Definition (Must Match Training!)
# ============================================
class SimpleSolarCNN(nn.Module):
    def __init__(self):
        super(SimpleSolarCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=9, stride=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.pool3 = nn.MaxPool2d(3, 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=2, stride=2)
        self.fc1 = nn.Linear(8 * 8 * 128, 4096)
        self.fc2 = nn.Linear(4096, 1)
        self.dropout_fc = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool3((F.relu(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout_fc(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ============================================
# Evaluation Function
# ============================================
def evaluate_model(model, test_loader, scaler, device):
    """Run inference and compute metrics"""
    model.eval()

    all_predictions = []
    all_targets = []

    print("\nRunning inference...")
    with torch.no_grad():
        for images, speeds in tqdm(test_loader, desc="Testing"):
            images = images.float().to(device)
            images = torch.log1p(torch.clamp(images, min=0))
            speeds = speeds.float().unsqueeze(1).to(device)

            outputs = model(images)

            all_predictions.extend(outputs.cpu().numpy().flatten())
            all_targets.extend(speeds.cpu().numpy().flatten())

    # Convert to arrays
    predictions = np.array(all_predictions)
    targets = np.array(all_targets)

    # Inverse transform to get original scale
    predictions_original = scaler.inverse_transform(predictions.reshape(-1, 1)).flatten()
    targets_original = scaler.inverse_transform(targets.reshape(-1, 1)).flatten()

    # Compute metrics
    mse = mean_squared_error(targets_original, predictions_original)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(targets_original, predictions_original)
    r2 = r2_score(targets_original, predictions_original)

    # Compute metrics on normalized values too
    mse_norm = mean_squared_error(targets, predictions)
    rmse_norm = np.sqrt(mse_norm)

    return {
        'predictions': predictions_original,
        'targets': targets_original,
        'predictions_norm': predictions,
        'targets_norm': targets,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mse_norm': mse_norm,
        'rmse_norm': rmse_norm,
    }

def plot_results(results, save_path='test_results.png'):
    """Plot predictions vs targets"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Scatter plot
    ax = axes[0]
    ax.scatter(results['targets'], results['predictions'], alpha=0.5, s=10)
    ax.plot([results['targets'].min(), results['targets'].max()],
            [results['targets'].min(), results['targets'].max()],
            'r--', lw=2, label='Perfect prediction')
    ax.set_xlabel('Actual Plasma Speed (km/s)')
    ax.set_ylabel('Predicted Plasma Speed (km/s)')
    ax.set_title(f'Predictions vs Actual\nR² = {results["r2"]:.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Time series plot
    ax = axes[1]
    n_samples = min(500, len(results['targets']))
    ax.plot(results['targets'][:n_samples], label='Actual', alpha=0.7)
    ax.plot(results['predictions'][:n_samples], label='Predicted', alpha=0.7)
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Plasma Speed (km/s)')
    ax.set_title('Time Series Comparison (first 500 samples)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅ Plot saved to {save_path}")
    plt.show()

# ============================================
# MAIN EXECUTION
# ============================================
if __name__ == '__main__':
    import multiprocessing
    try:
        multiprocessing.set_start_method('spawn', force=True)
    except RuntimeError:
        pass

    # ============================================
    # 1. Load 2018 Zarr Data
    # ============================================
    print("="*60)
    print(f"TESTING ON {TEST_YEAR} WITH {DELAY_DAYS}-DAY FIXED DELAY")
    print("="*60)

    print(f"\nLoading Zarr data for {TEST_YEAR}...")
    zarr_path = f"s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/{TEST_YEAR}"

    try:
        root = load_single_aws_zarr(path_to_zarr=zarr_path)
        data_2018 = root["193A"]
        print(f"✅ Loaded {data_2018.shape[0]} images")
        print(f"   Image shape: {data_2018.shape}")
    except Exception as e:
        print(f"❌ Failed to load from S3: {e}")
        raise

    # ============================================
    # 2. Load and Process Solar Wind Data
    # ============================================
    print(f"\nLoading solar wind data for {TEST_YEAR}...")
    df_plasma = pd.read_csv('omni2_2017.lst.txt',
                            sep=r'\s+',
                            header=None,
                            names=['year', 'day', 'hour', 'plasma_speed'])

    df_plasma['time'] = pd.to_datetime(
        df_plasma['year'].astype(str) + '-' + df_plasma['day'].astype(str) + ' ' +
        df_plasma['hour'].astype(str) + ':00:00',
        format='%Y-%j %H:%M:%S'
    )

    df_plasma.set_index('time', inplace=True)
    df_plasma['plasma_speed'] = df_plasma['plasma_speed'].replace(9999.0, np.nan)
    df_plasma['plasma_speed'] = df_plasma['plasma_speed'].interpolate(method='linear')

    print(f"✅ Loaded {len(df_plasma)} plasma records")
    print(f"   Date range: {df_plasma.index.min()} to {df_plasma.index.max()}")
    print(f"   Speed range: {df_plasma['plasma_speed'].min():.1f} - {df_plasma['plasma_speed'].max():.1f} km/s")

    # ============================================
    # 3. Apply Fixed 4-Day Delay Mapping
    # ============================================
    print(f"\nApplying fixed {DELAY_DAYS}-day delay...")

    # Get Zarr timestamps
    zarr_times_raw = data_2018.attrs['DATE-OBS']
    zarr_times_unsorted = pd.to_datetime(list(zarr_times_raw))

    sort_order = np.argsort(zarr_times_unsorted)
    zarr_times_sorted = zarr_times_unsorted[sort_order]

    # Apply fixed delay
    fixed_delay = pd.Timedelta(days=DELAY_DAYS)
    df_plasma['lookup_time'] = df_plasma.index - fixed_delay

    # Match to closest Zarr timestamp
    sorted_match_indices = get_nearest_sorted_indices(
        df_plasma['lookup_time'].values,
        zarr_times_sorted
    )

    df_plasma['zarr_idx'] = sort_order[sorted_match_indices]

    # Filter with 15-minute tolerance
    matched_timestamps = zarr_times_unsorted[df_plasma['zarr_idx'].values]
    time_diffs = np.abs(matched_timestamps - df_plasma['lookup_time'])

    df_test = df_plasma[time_diffs < pd.Timedelta(minutes=15)].copy()

    print(f"✅ Matched {len(df_test)} samples with fixed {DELAY_DAYS}-day delay")
    print(f"   Date range: {df_test.index.min()} to {df_test.index.max()}")

    # ============================================
    # 4. Cache Test Images (SAME STYLE AS TRAINING)
    # ============================================
    index_mapping = cache_images_for_year(df_test, data_2018, TEST_YEAR)

    # ============================================
    # 5. Load Checkpoint and Scaler
    # ============================================
    print("\nLoading trained model...")
    checkpoint_path = "best_model.pt"

    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"❌ Checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    # Extract scaler from checkpoint
    scaler = checkpoint['scaler']
    print(f"✅ Loaded scaler (trained range: {scaler.data_min_[0]:.1f} - {scaler.data_max_[0]:.1f} km/s)")

    # Scale test data using training scaler
    df_test_scaled = df_test.copy()
    df_test_scaled['plasma_speed'] = scaler.transform(df_test[['plasma_speed']])

    # ============================================
    # 6. Create Test Dataset and Loader
    # ============================================
    test_dataset = CachedSolarWindDataset(df_test_scaled, index_mapping, TEST_YEAR)

    test_loader = DataLoader(
        test_dataset,
        batch_size=128,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=False,
    )

    print(f"✅ Test dataset ready: {len(test_dataset)} samples")

    # ============================================
    # 7. Load Model and Run Inference
    # ============================================
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device}")

    model = SimpleSolarCNN()
    model.load_state_dict(checkpoint['model'])
    model.to(device)
    model.eval()

    print(f"✅ Model loaded from epoch {checkpoint['epoch']+1}")
    print(f"   Training val_loss: {checkpoint.get('val_loss', 'N/A')}")

    # Run evaluation
    results = evaluate_model(model, test_loader, scaler, device)

    # ============================================
    # 8. Print Results
    # ============================================
    print("\n" + "="*60)
    print("TEST RESULTS")
    print("="*60)
    print(f"Test samples: {len(results['targets'])}")
    print(f"Fixed delay: {DELAY_DAYS} days")
    print(f"\nMetrics (original scale):")
    print(f"  MSE:  {results['mse']:.2f} (km/s)²")
    print(f"  RMSE: {results['rmse']:.2f} km/s")
    print(f"  MAE:  {results['mae']:.2f} km/s")
    print(f"  R²:   {results['r2']:.4f}")
    print(f"\nMetrics (normalized scale):")
    print(f"  MSE:  {results['mse_norm']:.6f}")
    print(f"  RMSE: {results['rmse_norm']:.6f}")
    print("="*60)

    # ============================================
    # 9. Save Results
    # ============================================
    results_df = pd.DataFrame({
        'time': df_test.index,
        'actual_speed': results['targets'],
        'predicted_speed': results['predictions'],
        'error': results['targets'] - results['predictions'],
        'abs_error': np.abs(results['targets'] - results['predictions']),
    })

    results_df.to_csv(f'test_results_{TEST_YEAR}_delay{DELAY_DAYS}d.csv', index=False)
    print(f"\n✅ Results saved to test_results_{TEST_YEAR}_delay{DELAY_DAYS}d.csv")

    # Plot results
    plot_results(results, save_path=f'test_results_{TEST_YEAR}_delay{DELAY_DAYS}d.png')

    # Additional statistics
    print(f"\nError Statistics:")
    print(f"  Mean error: {results_df['error'].mean():.2f} km/s")
    print(f"  Std error:  {results_df['error'].std():.2f} km/s")
    print(f"  Max |error|: {results_df['abs_error'].max():.2f} km/s")
    print(f"  Median |error|: {results_df['abs_error'].median():.2f} km/s")

    print("\n✅ Testing complete!")

TESTING ON 2017 WITH 4-DAY FIXED DELAY

Loading Zarr data for 2017...
✅ Loaded 85277 images
   Image shape: (85277, 512, 512)

Loading solar wind data for 2017...
✅ Loaded 8760 plasma records
   Date range: 2017-01-01 00:00:00 to 2017-12-31 23:00:00
   Speed range: 264.0 - 817.0 km/s

Applying fixed 4-day delay...
✅ Matched 8528 samples with fixed 4-day delay
   Date range: 2017-01-05 00:00:00 to 2017-12-31 23:00:00

CACHING IMAGES FOR 2017

⏬ Downloading images for 2017...
   Total images to download: 8528


   2017:   0%|          | 0/9 [00:00<?, ?it/s]

KeyboardInterrupt: 